# Convert data from InSituPy into SpatialData

## Setup and Imports

In [1]:
# Enable autoreload for development
%load_ext autoreload
%autoreload 2

## Make sure `SpatialData` is installed

If it is not installed yet, install it with:
```bash
pip install spatialdata[extra]
```
Make sure the version is `>=0.7.2`. For more information on the installation of `SpatialData` see [here](https://spatialdata.scverse.org/en/stable/installation.html).

In [2]:
from pathlib import Path

from insitupy import CACHE, InSituData, InSituExperiment
from insitupy.spatialdata import convert_to_spatialdata

c:\Users\ge37voy\AppData\Local\miniconda3\envs\insitupy\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\ge37voy\AppData\Local\miniconda3\envs\insitupy\Lib\site-packages\spatialdata\_core\query\relational_query.py:531: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in enum.member() if you want to preserve the old behavior
  left = partial(_left_join_spatialelement_table)
c:\Users\ge37voy\AppData\Local\miniconda3\envs\insitupy\Lib\site-packages\spatialdata\_core\query\relational_query.py:532: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in enum.member() if you want to preserve the old behavior
  left_exclusive = partial(_left_exclusive_join_spatialelement_table)
c:\Users\ge37voy\AppData\Local\miniconda3\envs\

## Load InSituPy Data

First, let's load some example data. We'll demonstrate conversion with both:
- `InSituData`: A single spatial sample
- `InSituExperiment`: A collection of multiple samples

### Loading a Single Sample (InSituData)

In [3]:
# Load a single InSituData object
data_dir = Path(CACHE / "out/demo_insitupy_project")
xd = InSituData.read(data_dir)
xd.load_all()

In [4]:
# Display the InSituData object
xd

InSituData
Method:		Xenium
Slide ID:	0001879
Sample ID:	Replicate 1
Path:		C:\Users\ge37voy\.cache\InSituPy\out\demo_insitupy_project

    ➤ images
       'CD20':     (25778, 35416)
       'HE':       (25778, 35416, 3)
       'HER2':     (25778, 35416)
       'nuclei':   (25778, 35416)
    ➤ cells
       MultiCellData with main layer 'main'
           table
               AnnData object with n_obs × n_vars = 156447 × 297
               obs: 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'n_genes_by_counts', 'n_genes', 'leiden', 'cell_type_dc_sub_final', 'cell_type_publ'
               var: 'gene_ids', 'feature_types', 'genome', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
               uns: 'cell_type_dc_sub_final_colors', 'cell_type_publ_colors', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'umap'
               obsm: 'X_pca', 'X_umap', 'annotations', 'regions', 

### Create dataset with multiple samples (InSituExperiment)

**Note on Transcripts**: Due to dependency conflicts between `spatialdata` and `dask-geopandas`, transcript cropping with dask DataFrames becomes very inefficient when `spatialdata` is installed. The `dask-geopandas` library is used for efficient spatial filtering of large transcript datasets, but it cannot be used alongside `spatialdata`. 

To avoid performance issues when creating an `InSituExperiment` from regions (which involves cropping), we delete the transcripts beforehand.

> **Update**: This issue may be resolved in an upcoming version of `spatialdata`. The dask version has been unpinned in [PR #1006](https://github.com/scverse/spatialdata/pull/1006), which should allow `dask-geopandas` to be installed alongside `spatialdata` in a future release.

In [5]:
del xd.transcripts

In [6]:
exp = InSituExperiment.from_regions(
    data=xd,
    region_key="TMA"  # Column in annotations containing region IDs
)

In [7]:
# Display the experiment
exp

InSituExperiment (insitupy mode) with 6 samples:
           uid  CITAR slide_id    sample_id region_key region_name
0     92dc76ed  ++-++  0001879  Replicate 1        TMA         A-1
1     c9d5337d  ++-++  0001879  Replicate 1        TMA         A-2
2     77d55bd5  ++-++  0001879  Replicate 1        TMA         A-3
3     fda64ed4  ++-++  0001879  Replicate 1        TMA         B-1
4     9b82bec1  ++-++  0001879  Replicate 1        TMA         B-2
5     70c2bfd8  ++-++  0001879  Replicate 1        TMA         B-3

## Convert to SpatialData

The `convert_to_spatialdata()` function handles the conversion of all data modalities into SpatialData elements.

### Element naming convention

InSituPy uses a structured naming convention when converting to SpatialData:

**Single Sample (`InSituData`):**
```
MODALITY.key.subkey
```

Examples:
- `IMAGES.CD20` - CD20 staining image
- `CELLS.main.matrix` - Main cell expression matrix
- `CELLS.main.boundaries.cells` - Cell boundary masks
- `TRANSCRIPTS` - Transcript point data
- `ANNOTATIONS.Demo` - Annotation shapes

**Multiple Samples (`InSituExperiment`):**
```
SAMPLE.UID.MODALITY.key.subkey
```

When converting an `InSituExperiment`, each element is prefixed with the sample identifier to distinguish between samples:

Examples:
- `SAMPLE.1f1bcf1f.IMAGES.CD20` - CD20 image from Sample1
- `SAMPLE.1f1bcf1f.CELLS.main.matrix` - Cell matrix from Sample1
- `SAMPLE.1f1bcf1f.IMAGES.CD20` - CD20 image from Sample2
- `SAMPLE.1f1bcf1f.TRANSCRIPTS` - Transcripts from Sample2

### 3.1 Convert individual sample

In [8]:
# Convert InSituData to SpatialData
sdata = convert_to_spatialdata(xd)

2026-02-24 13:46:56 | [INFO] No case-insensitive conflicts found.


In [9]:
# Display the SpatialData object
sdata

SpatialData object
├── Images
│     ├── 'IMAGES.CD20': DataTree[cyx] (1, 25778, 35416), (1, 12889, 17708), (1, 6444, 8854), (1, 3222, 4427), (1, 1611, 2213), (1, 805, 1106)
│     ├── 'IMAGES.HE': DataTree[cyx] (3, 25778, 35416), (3, 12889, 17708), (3, 6444, 8854), (3, 3222, 4427), (3, 1611, 2213), (3, 805, 1106)
│     ├── 'IMAGES.HER2': DataTree[cyx] (1, 25778, 35416), (1, 12889, 17708), (1, 6444, 8854), (1, 3222, 4427), (1, 1611, 2213), (1, 805, 1106)
│     └── 'IMAGES.nuclei': DataTree[cyx] (1, 25778, 35416), (1, 12889, 17708), (1, 6444, 8854), (1, 3222, 4427), (1, 1611, 2213), (1, 805, 1106)
├── Labels
│     ├── 'CELLS.main.boundaries.cells': DataTree[yx] (25778, 35416), (12889, 17708), (6444, 8854), (3222, 4427), (1611, 2213), (805, 1106)
│     └── 'CELLS.main.boundaries.nuclei': DataTree[yx] (25778, 35416), (12889, 17708), (6444, 8854), (3222, 4427), (1611, 2213), (805, 1106)
├── Shapes
│     ├── 'ANNOTATIONS.Demo': GeoDataFrame shape: (28, 6) (2D shapes)
│     ├── 'ANNOTATIONS.Ja

### 3.2 Convert Experiment (Multiple Samples)

In [10]:
# Convert InSituExperiment to SpatialData
sdexp = convert_to_spatialdata(exp)

2026-02-24 13:47:11 | [INFO] No case-insensitive conflicts found.


In [11]:
# Display the experiment SpatialData
sdexp

SpatialData object
├── Images
│     ├── 'SAMPLE.9b82bec1..IMAGES.CD20': DataTree[cyx] (1, 4706, 4706), (1, 2353, 2353), (1, 1176, 1176), (1, 588, 588), (1, 294, 294), (1, 147, 147)
│     ├── 'SAMPLE.9b82bec1..IMAGES.HE': DataTree[cyx] (3, 4706, 4706), (3, 2353, 2353), (3, 1176, 1176), (3, 588, 588), (3, 294, 294), (3, 147, 147)
│     ├── 'SAMPLE.9b82bec1..IMAGES.HER2': DataTree[cyx] (1, 4706, 4706), (1, 2353, 2353), (1, 1176, 1176), (1, 588, 588), (1, 294, 294), (1, 147, 147)
│     ├── 'SAMPLE.9b82bec1..IMAGES.nuclei': DataTree[cyx] (1, 4706, 4706), (1, 2353, 2353), (1, 1176, 1176), (1, 588, 588), (1, 294, 294), (1, 147, 147)
│     ├── 'SAMPLE.70c2bfd8..IMAGES.CD20': DataTree[cyx] (1, 4706, 4706), (1, 2353, 2353), (1, 1176, 1176), (1, 588, 588), (1, 294, 294), (1, 147, 147)
│     ├── 'SAMPLE.70c2bfd8..IMAGES.HE': DataTree[cyx] (3, 4706, 4706), (3, 2353, 2353), (3, 1176, 1176), (3, 588, 588), (3, 294, 294), (3, 147, 147)
│     ├── 'SAMPLE.70c2bfd8..IMAGES.HER2': DataTree[cyx] (1, 4706, 

## Saving and Loading SpatialData

SpatialData objects can be saved to disk in Zarr format for efficient storage and lazy loading.

In [12]:
# Define output paths
outpath = CACHE / "test_spatialdata.zarr"
exp_outpath = CACHE / "exp_spatialdata.zarr"

In [13]:
# Save single sample SpatialData
sdata.write(outpath, overwrite=True)
print(f"Saved to: {outpath}")

c:\Users\ge37voy\AppData\Local\miniconda3\envs\insitupy\Lib\site-packages\ome_zarr\writer.py:319: FutureWarning: Passing storage-related arguments via **kwargs is deprecated. Please use the 'zarr_store_kwargs' parameter instead. **kwargs will be removed in a future version.
  da_delayed = da.to_zarr(


Saved to: C:\Users\ge37voy\.cache\InSituPy\test_spatialdata.zarr


In [14]:
# Save experiment SpatialData
sdexp.write(exp_outpath, overwrite=True)
print(f"Saved to: {exp_outpath}")

c:\Users\ge37voy\AppData\Local\miniconda3\envs\insitupy\Lib\site-packages\ome_zarr\writer.py:319: FutureWarning: Passing storage-related arguments via **kwargs is deprecated. Please use the 'zarr_store_kwargs' parameter instead. **kwargs will be removed in a future version.
  da_delayed = da.to_zarr(


Saved to: C:\Users\ge37voy\.cache\InSituPy\exp_spatialdata.zarr


### Load from Disk

In [15]:
from spatialdata import SpatialData

# Load saved SpatialData
sdata_loaded = SpatialData.read(outpath)
sdata_loaded

2026-02-24 13:50:17 | [INFO] root_attr: omero
2026-02-24 13:50:17 | [INFO] root_attr: version
2026-02-24 13:50:17 | [INFO] root_attr: multiscales
2026-02-24 13:50:17 | [INFO] datasets [{'path': '0', 'coordinateTransformations': [{'type': 'scale', 'scale': [1.0, 1.0, 1.0]}]}, {'path': '1', 'coordinateTransformations': [{'type': 'scale', 'scale': [1.0, 2.0, 2.0]}]}, {'path': '2', 'coordinateTransformations': [{'type': 'scale', 'scale': [1.0, 4.000310366232154, 4.0]}]}, {'path': '3', 'coordinateTransformations': [{'type': 'scale', 'scale': [1.0, 8.000620732464307, 8.0]}]}, {'path': '4', 'coordinateTransformations': [{'type': 'scale', 'scale': [1.0, 16.001241464928615, 16.003615002259377]}]}, {'path': '5', 'coordinateTransformations': [{'type': 'scale', 'scale': [1.0, 32.02236024844721, 32.02169981916818]}]}]
2026-02-24 13:50:17 | [INFO] resolution: 0
2026-02-24 13:50:17 | [INFO]  - shape ('c', 'y', 'x') = (1, 25778, 35416)
2026-02-24 13:50:17 | [INFO]  - chunks =  ['1', '4096 (+ 1202)', '

SpatialData object, with associated Zarr store: C:\Users\ge37voy\.cache\InSituPy\test_spatialdata.zarr
├── Images
│     ├── 'IMAGES.CD20': DataTree[cyx] (1, 25778, 35416), (1, 12889, 17708), (1, 6444, 8854), (1, 3222, 4427), (1, 1611, 2213), (1, 805, 1106)
│     ├── 'IMAGES.HE': DataTree[cyx] (3, 25778, 35416), (3, 12889, 17708), (3, 6444, 8854), (3, 3222, 4427), (3, 1611, 2213), (3, 805, 1106)
│     ├── 'IMAGES.HER2': DataTree[cyx] (1, 25778, 35416), (1, 12889, 17708), (1, 6444, 8854), (1, 3222, 4427), (1, 1611, 2213), (1, 805, 1106)
│     └── 'IMAGES.nuclei': DataTree[cyx] (1, 25778, 35416), (1, 12889, 17708), (1, 6444, 8854), (1, 3222, 4427), (1, 1611, 2213), (1, 805, 1106)
├── Labels
│     ├── 'CELLS.main.boundaries.cells': DataTree[yx] (25778, 35416), (12889, 17708), (6444, 8854), (3222, 4427), (1611, 2213), (805, 1106)
│     └── 'CELLS.main.boundaries.nuclei': DataTree[yx] (25778, 35416), (12889, 17708), (6444, 8854), (3222, 4427), (1611, 2213), (805, 1106)
├── Shapes
│     ├── '